# Waypoint — Data Fetch Demo

Demonstrates fetching historical return data using the built-in catalog and custom instruments.

**Prerequisites**
- `uv sync --extra dev` completed
- `.env` file in the repo root with your API keys (see `.env.example`)

```
FRED_API_KEY=your_key_here
EODHD_API_KEY=your_key_here
```

In [1]:
import waypoint as wp
import polars as pl
import numpy as np

## 1. Inspect catalog instruments

Each `Instrument` carries its display name, vendor symbol, vendor, frequency, and classification metadata.

In [2]:
catalog_instruments = [
    wp.catalog.equities.US_LARGE_CAP,
    wp.catalog.equities.US_SMALL_CAP,
    wp.catalog.equities.INTL_DEVELOPED,
    wp.catalog.fixed_income.US_AGG_BONDS,
    wp.catalog.fixed_income.US_TIPS,
    wp.catalog.indicators.REAL_RATE_10Y,
    wp.catalog.fixed_income.CPI_YOY,
]

pl.DataFrame([
    {
        "name": inst.name,
        "symbol": inst.symbol,
        "vendor": inst.vendor,
        "frequency": inst.frequency,
        "asset_class": inst.asset_class,
        "sub_asset_class": inst.sub_asset_class,
        "geography": inst.geography,
    }
    for inst in catalog_instruments
])

name,symbol,vendor,frequency,asset_class,sub_asset_class,geography
str,str,str,str,str,str,str
"""US Large Cap Equities""","""^SPX""","""yfinance""","""daily""","""Equities""","""Large Cap""","""US"""
"""US Small Cap Equities""","""^RUT""","""yfinance""","""daily""","""Equities""","""Small Cap""","""US"""
"""Intl Developed Equities""","""EFA""","""yfinance""","""daily""","""Equities""","""Developed""","""International"""
"""US Aggregate Bonds""","""AGG""","""yfinance""","""daily""","""Fixed Income""","""Aggregate""","""US"""
"""US TIPS""","""TIP""","""yfinance""","""daily""","""Fixed Income""","""Inflation-Linked""","""US"""
"""US 10-Year Real Rate""","""DFII10""","""fred""","""daily""","""Macro""","""Real Rates""","""US"""
"""CPI YoY""","""CPIAUCSL""","""fred""","""monthly""","""Macro""","""Inflation""","""US"""


## 2. Define a custom instrument

Use `wp.AssetDef` directly for anything not in the catalog.

In [3]:
GOLD = wp.AssetDef(
    name="Gold",
    symbol="GLD",
    vendor="yfinance",
    frequency="daily",
    asset_class="Alternatives",
    sub_asset_class="Commodities",
    geography="Global",
)

GOLD

AssetDef(name='Gold', symbol='GLD', vendor='yfinance', frequency=<Frequency.DAILY: 'daily'>, asset_class='Alternatives', sub_asset_class='Commodities', geography='Global')

## 3. Fetch a single equity instrument

For `frequency="daily"` instruments, the date range is automatically snapped to full calendar months so resampling to monthly is always clean.

In [4]:
spy = wp.fetch(wp.catalog.equities.US_LARGE_CAP, start="2020-01-01", end="2024-12-31")

print(f"Name      : {spy.name}")
print(f"Ticker    : {spy.ticker}")
print(f"Frequency : {spy.frequency}")
print(f"Asset class: {spy.asset_class} / {spy.sub_asset_class} / {spy.geography}")
print(f"Periods   : {len(spy.returns):,} observations")
spy.returns.head(10)

Name      : US Large Cap Equities
Ticker    : ^SPX
Frequency : daily
Asset class: Equities / Large Cap / US
Periods   : 1,257 observations


date,returns
date,f64
2020-01-03,-0.00706
2020-01-06,0.003533
2020-01-07,-0.002803
2020-01-08,0.004902
2020-01-09,0.006655
2020-01-10,-0.002855
2020-01-13,0.006976
2020-01-14,-0.001515
2020-01-15,0.00187


## 4. Fetch multiple instruments

Fetch a diversified set — equities, bonds, macro — and compare basic stats.

In [5]:
START = "2015-01-01"
END   = "2024-12-31"

assets = {
    inst.name: wp.fetch(inst, start=START, end=END)
    for inst in [
        wp.catalog.equities.US_LARGE_CAP,
        wp.catalog.equities.US_SMALL_CAP,
        wp.catalog.equities.INTL_DEVELOPED,
        wp.catalog.fixed_income.US_AGG_BONDS,
        GOLD,
    ]
}

for name, asset in assets.items():
    print(f"{name:<30} {len(asset.returns):>5} observations")

$^RUT: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)
$GLD: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)


US Large Cap Equities           2515 observations
US Small Cap Equities           2515 observations
Intl Developed Equities         2515 observations
US Aggregate Bonds              2515 observations
Gold                            2515 observations


## 5. Fetch a FRED macro series

Requires `FRED_API_KEY` in `.env`. Monthly CPI data.

In [6]:
cpi = wp.fetch(wp.catalog.fixed_income.CPI_YOY, start="2015-01-01", end="2024-12-31")

print(f"Name      : {cpi.name}")
print(f"Frequency : {cpi.frequency}")
print(f"Periods   : {len(cpi.returns)} observations")
cpi.returns.tail(12)

Name      : CPI YoY
Frequency : monthly
Periods   : 119 observations


date,returns
date,f64
2024-01-01,0.0031
2024-02-01,0.004098
2024-03-01,0.004431
2024-04-01,0.002171
2024-05-01,0.000486
…,…
2024-08-01,0.001572
2024-09-01,0.002133
2024-10-01,0.002856


## 6. Quick return statistics

Show annualized mean and std for each fetched daily asset.

In [7]:
rows = []
for name, asset in assets.items():
    ppy = asset.periods_per_year
    returns = asset.returns["returns"]
    ann_return = float((1 + returns).product() ** (ppy / len(returns)) - 1)
    ann_vol    = float(returns.std() * np.sqrt(ppy))
    rows.append({
        "instrument": name,
        "observations": len(returns),
        "ann_return_%": round(ann_return * 100, 2),
        "ann_vol_%":    round(ann_vol    * 100, 2),
        "sharpe (rf=0)": round(ann_return / ann_vol, 2) if ann_vol else None,
    })

pl.DataFrame(rows)

instrument,observations,ann_return_%,ann_vol_%,sharpe (rf=0)
str,i64,f64,f64,f64
"""US Large Cap Equities""",2515,11.09,17.83,0.62
"""US Small Cap Equities""",2515,6.42,22.92,0.28
"""Intl Developed Equities""",2515,5.26,17.28,0.3
"""US Aggregate Bonds""",2515,1.29,5.31,0.24
"""Gold""",2515,7.83,14.12,0.55


## 7. Force-refresh cached data

Use `force_refresh=True` to bypass the cache and re-fetch from the vendor (e.g. after a corporate action adjustment is published).

In [8]:
spy_fresh = wp.fetch(wp.catalog.equities.US_LARGE_CAP, start="2024-01-01", end="2024-12-31", force_refresh=True)
print(f"Re-fetched {len(spy_fresh.returns)} observations for {spy_fresh.name}")

Re-fetched 251 observations for US Large Cap Equities
